# 💬 LLM APIs & Prompting: Zero to Hero — A Guided Lab

Learn to program Large Language Models (like GPT and Gemini) reliably: structured output,
prompt patterns, conversation state, cost control, and robust error handling.

**Runs 100% offline.** We use a `MockLLM` with the *same interface* as a real client, so you
learn the patterns without API keys or cost. Swapping in OpenAI/Gemini later is a one-line change.

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ ASCII diagram → 🔬 Worked example →
⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. How an LLM API call actually works
2. Roles: system / user / assistant
3. Sampling parameters (temperature, top_p)
4. Prompt patterns: zero-shot, few-shot, chain-of-thought
5. Structured output (reliable JSON)
6. Conversation state & context windows
7. Cost & token budgeting
8. Error handling & retries
9. Putting it together: a mini application
10. 🏆 Capstone: a robust ticket-triage pipeline


In [ ]:
import json, time, random, re

class MockLLM:
    """A stand-in for a real LLM API (OpenAI/Gemini) with the same shape.
    Replace with a real client that exposes .chat(messages, temperature) later."""
    def chat(self, messages, temperature=0.7, max_tokens=256):
        # 'messages' is a list of {"role","content"} dicts, like real APIs
        user = " ".join(m["content"] for m in messages if m["role"]=="user").lower()
        # crude deterministic behaviors so exercises are checkable
        if "json" in user and "sentiment" in user:
            s = "positive" if "love" in user or "great" in user else ("negative" if "broke" in user or "terrible" in user else "neutral")
            return json.dumps({"sentiment": s, "confidence": 0.9})
        if "classify" in user or "category" in user:
            for cat in ["billing","technical","account","shipping"]:
                if cat in user: return cat
            return "other"
        if "step by step" in user or "reason" in user:
            return "Step 1: understand. Step 2: compute. Final answer: 42."
        return "This is a mock completion. (Plug in a real LLM for real answers.)"

llm = MockLLM()
print(llm.chat([{"role":"user","content":"Classify this billing issue"}]))

---
## Chapter 1 — How an LLM API Call Works

📖 **Theory.** You send a **list of messages** (each with a `role` and `content`); the model
returns a **completion** (text). It's stateless — the API doesn't remember previous calls, so
*you* resend the whole conversation each time.

🖼️ **Diagram — request/response**
```
 YOUR CODE                          LLM API
 ┌──────────────┐   messages[]     ┌───────────┐
 │ system: ...  │  ─────────────►  │  model    │
 │ user: ...    │                  │ generates │
 └──────────────┘  ◄─────────────  │ next text │
        completion text            └───────────┘
```

🧠 **Mental model.** An LLM is a very powerful autocomplete: given the conversation so far, it
predicts the most likely continuation. Everything else (JSON, reasoning, roleplay) is you
*shaping* that autocomplete with good prompts.


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Classify this ticket into a category: my card was charged twice"},
]
response = llm.chat(messages)
print("response:", response)

### ✏️ Your Turn 1.1
Send a message asking the model to classify a **shipping** problem (e.g. "my package never
arrived"). Confirm you get back `"shipping"` from the mock.

In [ ]:
# build messages, call llm.chat, print result
result = None
print(result)

✅ **Solution**
```python
msgs = [{"role":"user","content":"classify this: my shipping package never arrived"}]
result = llm.chat(msgs)   # "shipping"
```

---
## Chapter 2 — Roles: system / user / assistant

📖 **Theory.** Three roles shape the conversation:
- **system** — sets behavior/persona/rules (the "job description"). Highest-level instructions.
- **user** — the human's input.
- **assistant** — the model's previous replies (you include these to give it memory).

🖼️ **Diagram — role stack**
```
 system     ── "You are a terse SQL expert. Only output SQL."   (persona/rules)
 user       ── "users older than 30"
 assistant  ── "SELECT * FROM users WHERE age > 30;"            (prior reply)
 user       ── "...now only their emails"                       (follow-up)
```

⚡ **Pro tip.** Put durable rules (tone, format, constraints) in the **system** message, not
repeated in every user turn. It's more reliable and cheaper.


In [ ]:
def ask(system, user):
    return llm.chat([
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ])

print(ask("You are a ticket classifier. Reply with one word.",
          "The app crashes when I open it — technical issue?"))

### ✏️ Your Turn 2.1
Write a `system` prompt that makes the assistant behave as a "polite refund-policy bot", then
ask it a user question. (With the mock, focus on *crafting* the messages correctly.)

In [ ]:
# system + user, call ask()


✅ **Solution**
```python
print(ask("You are a polite refund-policy assistant. Be concise and friendly.",
          "Can I return an item after 20 days?"))
```

---
## Chapter 3 — Sampling Parameters (temperature & top_p)

📖 **Theory.** LLMs pick the next token from a probability distribution. Parameters shape it:
- **temperature** (0–2): low = focused/deterministic; high = random/creative.
- **top_p** (0–1): nucleus sampling — only consider the smallest set of tokens whose
  probabilities sum to `p`.

🖼️ **Diagram — temperature reshapes the distribution**
```
 temp=0.2 (sharp)          temp=1.5 (flat)
   █                          █ █ █
   █ ▁ ▁ ▁                    █ █ █ ▄ ▄
   pick top almost always     spread out, more variety
```

🧠 **Mental model.** Temperature is a "creativity dial." Facts/extraction → low (0–0.3).
Brainstorming/creative writing → high (0.7–1.0).


In [ ]:
# Our mock ignores temperature, but this is the REAL call signature you'd use:
def generate(prompt, temperature=0.7):
    return llm.chat([{"role":"user","content":prompt}], temperature=temperature)

print("low temp (deterministic tasks):", generate("Extract the total from: Invoice $42", temperature=0.0))
print("high temp (creative tasks):    ", generate("Write a slogan", temperature=1.0))

### ✏️ Your Turn 3.1
For each task, state the temperature you'd choose and why (write it as a comment):
(a) extracting a date from an email, (b) generating 10 creative product names,
(c) summarizing a legal document.

In [ ]:
# (a) ... temperature = ?  because ...
# (b) ...
# (c) ...


✅ **Solution**
```python
# (a) 0.0-0.2  -> must be exact/deterministic
# (b) 0.8-1.0  -> want variety/creativity
# (c) 0.0-0.3  -> want faithful, non-embellished summary
```

---
## Chapter 4 — Prompt Patterns

📖 **Theory.** Three foundational patterns:
- **Zero-shot** — just ask. ("Classify this review's sentiment.")
- **Few-shot** — show a few input→output examples first; the model imitates the pattern.
- **Chain-of-thought (CoT)** — ask it to "think step by step" before answering, for reasoning
  tasks.

🖼️ **Diagram — few-shot primes the format**
```
 Review: "Loved it!"     -> positive
 Review: "Broke fast"    -> negative
 Review: "It's okay"     -> neutral
 Review: "Best purchase" -> ?        ◄── model continues the pattern
```

⚡ **Pro tip.** Few-shot examples are often *more reliable* than long instructions for locking
in an output format.


In [ ]:
# Few-shot classification prompt
few_shot = """Classify the sentiment as positive, negative, or neutral.

Review: "Absolutely love this, works great!"
Sentiment: positive

Review: "It broke after two days, terrible."
Sentiment: negative

Review: "It is fine, nothing special."
Sentiment: neutral

Review: "Best purchase I have made all year!"
Sentiment:"""
print("few-shot ->", llm.chat([{"role":"user","content":few_shot}]))

# Chain-of-thought
cot = "A store had 100 items, sold 30, restocked 20. How many now? Think step by step."
print("CoT ->", llm.chat([{"role":"user","content":cot}]))

⚠️ **Common trap.** Chain-of-thought helps *reasoning* tasks but wastes tokens on simple
lookups. Don't add "think step by step" to a trivial classification.

### ✏️ Your Turn 4.1
Write a **few-shot** prompt that classifies support tickets into `billing / technical /
shipping`, with 3 examples, then a 4th ticket to classify.

In [ ]:
fewshot_tickets = None
print(llm.chat([{"role":"user","content":fewshot_tickets}]) if fewshot_tickets else None)

✅ **Solution**
```python
fewshot_tickets = """Classify into billing, technical, or shipping.

Ticket: "I was double charged" -> billing
Ticket: "App won\'t load" -> technical
Ticket: "Package is late" -> shipping
Ticket: "My payment failed" ->"""
```

---
## Chapter 5 — Structured Output (reliable JSON)

📖 **Theory.** Apps need *parseable* output, not prose. The recipe:
1. **Explicitly** instruct: "Respond with ONLY valid JSON, no prose."
2. Specify the exact schema (keys and value types).
3. **Parse defensively** — wrap `json.loads` in try/except; never trust it's perfect.

🖼️ **Diagram — the parse-safely pattern**
```
 prompt("...ONLY JSON {sentiment, confidence}...")
        │
        ▼  raw text
   try: json.loads(text) ─► dict ✅
   except: repair / retry / fallback ⚠️
```


In [ ]:
def extract_sentiment(review):
    prompt = (
        "Analyze sentiment. Respond with ONLY valid JSON: "
        '{"sentiment": "positive|negative|neutral", "confidence": 0.0-1.0}. '
        f"No other text.\n\nReview: {review}"
    )
    raw = llm.chat([{"role":"user","content":prompt}])
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"sentiment": "unknown", "confidence": 0.0, "error": "unparseable"}

print(extract_sentiment("I love this, it's great!"))
print(extract_sentiment("The product broke, terrible experience"))

⚠️ **Common trap.** Real models sometimes wrap JSON in markdown fences (```json ... ```) or
add a sentence before it. Defensive parsing (strip fences, try/except) is mandatory in
production.

### ✏️ Your Turn 5.1
Write `extract_fields(text)` that asks for JSON with keys `name`, `email`, `amount` and
returns a parsed dict (with a safe fallback on parse failure).

In [ ]:
def extract_fields(text):
    pass
print(extract_fields("Invoice for John, john@x.com, $42"))

✅ **Solution**
```python
def extract_fields(text):
    prompt = ('Extract as ONLY JSON {"name":..,"email":..,"amount":..}. '
              f"Text: {text}")
    raw = llm.chat([{"role":"user","content":prompt}])
    try: return json.loads(raw)
    except json.JSONDecodeError: return {"error": "parse_failed", "raw": raw}
```

---
## Chapter 6 — Conversation State & Context Windows

📖 **Theory.** APIs are **stateless** — to give the model memory, you resend the growing
message list. But the **context window** (max tokens) is finite and costs money, so long
chats need a strategy: **truncate** old turns or **summarize** them.

🖼️ **Diagram — sliding + summary memory**
```
 [sys][u1][a1][u2][a2][u3][a3][u4]...   ← grows forever (bad)
         │ summarize old turns
         ▼
 [sys][summary of u1..a2][u3][a3][u4]   ← bounded (good)
```


In [ ]:
class Conversation:
    def __init__(self, system, max_turns=6):
        self.messages = [{"role":"system","content":system}]
        self.max_turns = max_turns
    def add(self, role, content):
        self.messages.append({"role":role,"content":content})
        self._trim()
    def _trim(self):
        # keep system + the last max_turns messages
        if len(self.messages) > self.max_turns + 1:
            system = self.messages[0]
            recent = self.messages[-self.max_turns:]
            self.messages = [system] + recent
    def send(self, user_text):
        self.add("user", user_text)
        reply = llm.chat(self.messages)
        self.add("assistant", reply)
        return reply

chat = Conversation("You are a support bot.", max_turns=4)
chat.send("I have a billing question")
chat.send("classify: I was charged twice")
print("current message count:", len(chat.messages))
print("last reply:", chat.messages[-1]["content"])

### ✏️ Your Turn 6.1
Add a method `summary_trim()` to `Conversation` that, when over the limit, replaces the old
middle turns with a single system note `"[earlier conversation summarized]"` while keeping the
last 2 turns verbatim.

In [ ]:
# extend Conversation or write a function that does summary-based trimming


✅ **Solution**
```python
def summary_trim(conv):
    if len(conv.messages) > conv.max_turns + 1:
        system = conv.messages[0]
        recent = conv.messages[-2:]
        note = {"role":"system","content":"[earlier conversation summarized]"}
        conv.messages = [system, note] + recent
```

---
## Chapter 7 — Cost & Token Budgeting

📖 **Theory.** APIs bill per **token** (~¾ of a word). Cost = (input_tokens + output_tokens) ×
price. Long prompts and long histories cost real money and can degrade attention. Estimate and
budget deliberately.

🧠 **Mental model.** Every word in your prompt (and the whole resent history) is on the meter.
Trim ruthlessly; summarize old turns; cap `max_tokens`.


In [ ]:
def estimate_tokens(text):
    # rough heuristic: ~1 token per 4 characters (real: use tiktoken)
    return max(1, len(text) // 4)

def estimate_cost(messages, price_per_1k=0.002):
    total = sum(estimate_tokens(m["content"]) for m in messages)
    return total, (total/1000)*price_per_1k

msgs = [{"role":"system","content":"You are a helpful assistant."},
        {"role":"user","content":"Summarize this very long document " * 20}]
tokens, cost = estimate_cost(msgs)
print(f"~{tokens} tokens, ~${cost:.5f} per call")

⚡ **Pro tip.** In production use the model's real tokenizer (e.g. `tiktoken` for OpenAI) for
accurate counts. The `/4` heuristic is only for rough budgeting.

### ✏️ Your Turn 7.1
Compare the estimated token count of a short prompt vs. a long one, and compute how many times
cheaper the short one is.

In [ ]:
short = None; long = None
ratio = None
print(ratio)

✅ **Solution**
```python
short = estimate_tokens("Summarize this.")
long = estimate_tokens("Summarize this document in exhaustive detail. " * 30)
ratio = long / short
```

---
## Chapter 8 — Error Handling & Retries

📖 **Theory.** Real APIs fail transiently: rate limits (429), timeouts, network blips. Robust
code **retries with exponential backoff** and gives up gracefully after N attempts.

🖼️ **Diagram — retry with backoff**
```
 try call ──fail──► wait 1s ──► retry ──fail──► wait 2s ──► retry ──fail──► wait 4s ──► raise
     │
   success ──► return
```


In [ ]:
class FlakyLLM(MockLLM):
    def __init__(self, fail_times=2):
        self.calls = 0; self.fail_times = fail_times
    def chat(self, messages, temperature=0.7, max_tokens=256):
        self.calls += 1
        if self.calls <= self.fail_times:
            raise TimeoutError("simulated rate limit / timeout")
        return super().chat(messages, temperature, max_tokens)

def call_with_retry(client, messages, max_retries=4, base_delay=0.05):
    for attempt in range(max_retries):
        try:
            return client.chat(messages)
        except TimeoutError as e:
            if attempt == max_retries - 1:
                raise
            delay = base_delay * (2 ** attempt)   # exponential backoff
            print(f"  attempt {attempt+1} failed ({e}); retrying in {delay:.2f}s")
            time.sleep(delay)

flaky = FlakyLLM(fail_times=2)
print("result:", call_with_retry(flaky, [{"role":"user","content":"hello"}]))

### ✏️ Your Turn 8.1
Modify `call_with_retry` to also accept a `fallback` value returned instead of raising when all
retries are exhausted (so the app degrades gracefully rather than crashing).

In [ ]:
def call_with_fallback(client, messages, max_retries=3, fallback="[unavailable]"):
    pass
print(call_with_fallback(FlakyLLM(fail_times=99), [{"role":"user","content":"hi"}]))

✅ **Solution**
```python
def call_with_fallback(client, messages, max_retries=3, fallback="[unavailable]"):
    for attempt in range(max_retries):
        try: return client.chat(messages)
        except TimeoutError:
            if attempt == max_retries-1: return fallback
            time.sleep(0.05*(2**attempt))
```

---
## Chapter 9 — Putting It Together: a Mini Application

📖 **Theory.** Real LLM features combine everything: a system prompt, structured output,
parsing, and error handling — wrapped in a clean function. Let's build an **auto-tagger** that
returns a category + priority as JSON.


In [ ]:
def auto_tag_ticket(text):
    prompt = (
        "You are a support triage system. Classify the ticket and assign priority.\n"
        'Respond with ONLY JSON: {"category": "billing|technical|shipping|account|other", '
        '"priority": "low|medium|high"}.\n\n'
        f"Ticket: {text}"
    )
    messages = [{"role":"system","content":"You output only JSON."},
                {"role":"user","content":prompt}]
    try:
        raw = call_with_retry(MockLLM(), messages)
        # mock returns a bare category; wrap into the expected schema for the demo
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return {"category": raw.strip(), "priority": "medium"}
    except Exception:
        return {"category": "other", "priority": "medium", "error": "llm_unavailable"}

for t in ["I was charged twice for one order",
          "The website is completely down",
          "Where is my package?"]:
    print(t, "->", auto_tag_ticket(t))

### ✏️ Your Turn 9.1
Extend `auto_tag_ticket` to also include a `"needs_human"` boolean that is True when the
category is `billing` (money issues often need a human). Return it in the dict.

In [ ]:
# modify auto_tag_ticket to add needs_human


✅ **Solution**
```python
tag = auto_tag_ticket(text)
tag["needs_human"] = (tag.get("category") == "billing")
```

---
## 🏆 Chapter 10 — Capstone: Robust Ticket-Triage Pipeline

Build a production-shaped function `triage(ticket)` that:
1. Uses a clear **system** prompt + a **few-shot** user prompt.
2. Requests **structured JSON** (`category`, `priority`, `summary`).
3. **Parses defensively** (fenced JSON, or fallback).
4. Uses **retry with fallback** so it never crashes.
5. Returns a clean dict ready for the rest of an app.

Attempt it before revealing the solution.

In [ ]:
# Your triage() implementation here
def triage(ticket):
    pass

# test cases
for t in ["Double charged on my Visa", "App crashes on login", "Package lost in transit"]:
    print(t, "->", triage(t))

✅ **Capstone Solution**
```python
def triage(ticket):
    system = "You are an automated support triage engine. You output only valid JSON."
    user = (
        "Classify into category (billing|technical|shipping|account|other), "
        "priority (low|medium|high), and give a one-line summary.\n"
        'Format: {"category":..,"priority":..,"summary":..}\n\n'
        "Examples:\n"
        'Ticket: "charged twice" -> {"category":"billing","priority":"high","summary":"duplicate charge"}\n'
        f'Ticket: "{ticket}" ->'
    )
    messages = [{"role":"system","content":system},{"role":"user","content":user}]

    def parse(raw):
        raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try: return json.loads(raw)
        except json.JSONDecodeError:
            return {"category": raw.split()[0] if raw else "other",
                    "priority":"medium", "summary": ticket[:40]}

    for attempt in range(3):
        try:
            return parse(MockLLM().chat(messages))
        except Exception:
            if attempt == 2:
                return {"category":"other","priority":"medium",
                        "summary":ticket[:40], "error":"llm_unavailable"}
            time.sleep(0.05*(2**attempt))
```

🎉 **You can now program LLMs like an engineer**, not just chat with them: roles, sampling,
prompt patterns, structured output, memory, cost, and resilience. Swap `MockLLM` for a real
OpenAI/Gemini client (same `.chat(messages)` shape) and everything works.

---
### 📌 Real-client swap (for reference)
```python
# OpenAI (pip install openai)
from openai import OpenAI
client = OpenAI(api_key="...")
def real_chat(messages, temperature=0.7):
    r = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=temperature)
    return r.choices[0].message.content

# Google Gemini (pip install google-generativeai)
import google.generativeai as genai
genai.configure(api_key="...")
gm = genai.GenerativeModel("gemini-1.5-flash")
def gemini_chat(prompt):
    return gm.generate_content(prompt).text
```

### 📌 Function/Concept Quick-Reference
**Messages:** roles system/user/assistant, `chat(messages, temperature, max_tokens)`
**Sampling:** temperature (low=deterministic, high=creative), top_p
**Patterns:** zero-shot, few-shot, chain-of-thought
**Structured output:** explicit "ONLY JSON" + schema + defensive `json.loads`
**Memory:** resend history, truncate/summarize for context window
**Cost:** tokens ≈ chars/4, budget input+output
**Resilience:** try/except, exponential backoff, fallback values
